In [1]:
from pathlib import Path
import torch
import pandas as pd
from IPython.display import display
from transformers import AutoTokenizer

torch.set_num_threads(2)
root = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if p.name == "miniproject2" and (p / "data/train.csv").is_file()), None)
if root is None:
    root = Path.cwd() / "miniproject2"
if not (root / "data/train.csv").is_file():
    raise FileNotFoundError("miniproject2 또는 note 폴더에서 실행하세요.")
# 기존 캐시가 있으면 재사용. 없으면 새 폴더에 모델을 다운로드합니다.
shared_cache = root.parent / "Note/miniproject1/.model_cache"
cache_dir = shared_cache if shared_cache.is_dir() else root / "note/.model_cache"
local_only = shared_cache.is_dir()
from transformers import AutoModelForSeq2SeqLM
model_id = "theSOL1/kogrammar-base"
tokenizer = AutoTokenizer.from_pretrained(model_id, cache_dir=str(cache_dir), local_files_only=local_only)
model = AutoModelForSeq2SeqLM.from_pretrained(model_id, cache_dir=str(cache_dir), local_files_only=local_only).eval()
print("모델:", model_id)
print("클래스:", type(model).__name__)


모델: theSOL1/kogrammar-base
클래스: BartForConditionalGeneration


In [2]:
display(pd.DataFrame([{"구조": "encoder–decoder", "encoder 층": model.config.encoder_layers,
                       "decoder 층": model.config.decoder_layers, "hidden 차원": model.config.d_model,
                       "어휘 수": model.config.vocab_size}]))
print("출력 head:", model.lm_head)
print("문장 종료 토큰:", tokenizer.eos_token, tokenizer.eos_token_id)


,구조,encoder 층,decoder 층,hidden 차원,어휘 수
0,encoder–decoder,6,6,768,30000


출력 head: Linear(in_features=768, out_features=30000, bias=False)
문장 종료 토큰: </s> 1


In [3]:
text = "비가 와서 우산을 챙겻다."
inputs = tokenizer(text, return_tensors="pt")
ids = inputs["input_ids"][0].tolist()
display(pd.DataFrame({"위치": range(len(ids)), "입력 토큰": tokenizer.convert_ids_to_tokens(ids), "토큰 ID": ids}))


,위치,입력 토큰,토큰 ID
0,0,▁비가,18844
1,1,▁와서,18799
2,2,▁우,14093
3,3,산을,16142
4,4,▁챙,16848
5,5,겻,9083
6,6,다.,14130


In [4]:
with torch.inference_mode():
    output_ids = model.generate(**inputs, max_new_tokens=128, num_beams=4, do_sample=False)
corrected = tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()
print("입력:", text)
print("생성 토큰 ID:", output_ids[0].tolist())
print("교정 출력:", corrected)


입력: 비가 와서 우산을 챙겻다.
생성 토큰 ID: [1, 18844, 18799, 14093, 16142, 16848, 9084, 14130, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 1]
교정 출력: 비가 와서 우산을 챙겼다.


In [5]:
examples = [
    ("비가 와서 우산을 챙겻다.", "비가 와서 우산을 챙겼다."),
    ("오늘은 날씨가 조아서 산책을 햇다.", "오늘은 날씨가 좋아서 산책을 했다."),
    ("오늘은 날씨가 좋아서 산책을 했다.", "오늘은 날씨가 좋아서 산책을 했다."),
]
batch = tokenizer([x for x, _ in examples], padding=True, return_tensors="pt")
with torch.inference_mode():
    outputs = model.generate(**batch, max_new_tokens=128, num_beams=4, do_sample=False)
records = []
for (source, target), pred in zip(examples, tokenizer.batch_decode(outputs, skip_special_tokens=True)):
    pred = pred.strip()
    records.append({"입력": source, "교정 출력": pred, "기대 정답": target, "정답 일치": pred == target})
display(pd.DataFrame(records))


,입력,교정 출력,기대 정답,정답 일치
0,비가 와서 우산을 챙겻다.,비가 와서 우산을 챙겼다.,비가 와서 우산을 챙겼다.,True
1,오늘은 날씨가 조아서 산책을 햇다.,오늘은 날씨가 조아서 산책을 했다.,오늘은 날씨가 좋아서 산책을 했다.,False
2,오늘은 날씨가 좋아서 산책을 했다.,오늘은 날씨가 좋아서 산책을 했다.,오늘은 날씨가 좋아서 산책을 했다.,True


In [6]:
masked_text = "비가 와서 [MASK]을 챙겻다."
masked_inputs = tokenizer(masked_text, return_tensors="pt")
with torch.inference_mode():
    masked_output = model.generate(**masked_inputs, max_new_tokens=128, num_beams=4, do_sample=False)
masked_result = tokenizer.decode(masked_output[0], skip_special_tokens=True).strip()
print("입력:", masked_text)
print("교정 출력:", masked_result)
print("빈칸 표시가 그대로 하나 유지됨:", masked_result.count("[MASK]") == 1)


입력: 비가 와서 [MASK]을 챙겻다.
교정 출력: 비가 와서 [MASK]을 챙겼다.
빈칸 표시가 그대로 하나 유지됨: True
